# Hotel Booking Analysis 

## 1. Project Overview

The analysis done with this project analyzes hotel booking data to identify patterns in bookings, cancellations, pricing, and estimated booking value. 

The use of Python and pandas for this analysis was to explore the behavior of bookings from both City Hotel and Resort Hotel, and to identify factors that may affect the hotel performance. 

### Business Questions

The analysis focuses on the following:

- How do City Hotel and Resort Hotel distinct from one another in regard to booking volume?
- Booking cancellations are associated with what factors?
- In what way does lead time differ from the canceled and completed bookings?
- How does the Average Daily Rate (ADR) vary by the month and hotel?
- Which month brought in the highest booking volume?
- How does estimated booking value diffe from both hotels?
- Which months, and hotel, show the strongest estimated value from completed bookings?

### Tools Used
- Python
- pandas
- Jupyter Notebook
- Visual Studio Code

### Dataset

The following data set contains hotel booking records with information regarding hotel type, booking status, arrival date, lead time, length of stay, ADR, distribution channels, customer information, and any addtional booking characteristics. 

### Important Metric Note

ADR x total nights

They are not to be interperted as official accounting revenue. Completed-booking values exclude canceled bookings; however, the metric remains an estimated room value based on the available booking data. 

In [56]:
import pandas as pd
import numpy as np

In [57]:
df = pd.read_csv("../data/hotel_bookings.csv")

## 4. Initial Data Exploration 

Before performing any analysis the dataset was inspected to understand its size, structure, columns, and any potential data-quality issues.

In [58]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [59]:
df.shape

(119390, 32)

In [60]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')

## 5. Data Quality & Missing Values

The following dataset was inspected for any missing values to determine which fields require cleaning or any special handling during the analyis

In [61]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [62]:
missing = df.isnull().sum()
missing[missing > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [63]:
df[df["adr"] < 0][["hotel", "adr", "is_canceled"]]

,hotel,adr,is_canceled
14969,Resort Hotel,-6.38,0


In [64]:
(df["adr"] < 0).sum()

np.int64(1)

In [65]:
df[df["adr"] > 1000][
    ["hotel", "adr", "is_canceled", "market_segment"]
]

,hotel,adr,is_canceled,market_segment
48515,City Hotel,5400.0,1,Offline TA/TO


## 6. Agent Analysis 

The agent field is identified as the travel agent associated with a booking. This analysis examines agent activity, booking volume, and cancellation behavior. 

In [66]:
df["agent"].isna().value_counts()

agent
False    103050
True      16340
Name: count, dtype: int64

In [67]:
top_agents = df["agent"].value_counts().head(10)
top_agents

agent
9.0      31961
240.0    13922
1.0       7191
14.0      3640
7.0       3539
6.0       3290
250.0     2870
241.0     1721
28.0      1666
8.0       1514
Name: count, dtype: int64

In [68]:
top_agents_percentage = (
    top_agents / df["agent"].notna().sum() * 100
).round(2)

top_agents_percentage

agent
9.0      31.02
240.0    13.51
1.0       6.98
14.0      3.53
7.0       3.43
6.0       3.19
250.0     2.79
241.0     1.67
28.0      1.62
8.0       1.47
Name: count, dtype: float64

In [69]:
top_agents.sum()

np.int64(71314)

In [70]:
agent_cancellation = (
    df.groupby("agent")["is_canceled"]
      .agg(["count", "mean"])
      .sort_values("count", ascending=False)
      .head(10)
)

agent_cancellation

,count,mean
agent,,
9.0,31961,0.415006
240.0,13922,0.393909
1.0,7191,0.734251
14.0,3640,0.179121
7.0,3539,0.133936
6.0,3290,0.311550
250.0,2870,0.178746
241.0,1721,0.137130
28.0,1666,0.066026


### Agent Analysis Finding

Then top ten agents are considered for exactly 69.2% of  bookings with assigned agent, indicating that booking activity was very concentrated among a small team of agents.

Cancellation bahevior varied considerably between high-volume agents. An example would be, Agent 1 had a cancellation rate of approximately 73.4%.

This advises that the booking volume and booking quality were not necessarily aligned across agents and may need further investigation. 

## 7. Cancellation Analysis 

It is important to note that booking cancellations are an important factor of the hotel performance, because canceled reservations are able to affect occupancy planning, revenue expectations, and operational decisions. 

This analysis compares the canceled and completed bookings and examines just how cancellation behavior differs between hotel types.

In [71]:
cancellation_summary = (
    df.groupby("is_canceled")["lead_time"]
      .agg(["count", "mean", "median"])
)

cancellation_summary

,count,mean,median
is_canceled,,,
0,75166,79.984687,45.0
1,44224,144.848815,113.0


In [72]:
cancellation_by_hotel = (
    df.groupby("hotel")["is_canceled"]
      .mean()
      .sort_values(ascending=False)
)

cancellation_by_hotel

hotel
City Hotel      0.417270
Resort Hotel    0.277634
Name: is_canceled, dtype: float64

In [73]:
hotel_cancellation = (
    df.groupby(["hotel", "is_canceled"])["lead_time"]
      .agg(["count", "mean", "median"])
)

hotel_cancellation

count        mean  median
hotel        is_canceled                           
City Hotel   0            46228   80.702734    50.0
             1            33102  150.281222   115.0
Resort Hotel 0            28938   78.837618    38.0
             1            11122  128.680543   109.0

### Cancellation Analysis Finding

The canceled bookings had a significant longer lead time than those of completed bookings. The average lead time was exactly 145 days for canceled bookings compared with approximately 80 days for bookings that were not canceled.

Cancellation rates do not coincide with hotel types- they differ significantly, if anything. City Hotel had a cancellation rate of exactly 41.7% compared with approximately 27.8& for Resort Hotel. 

In both hotel types, canceled bookings had a longer average lead times than completed bookings. This could indicated that reservations made way inadvance could hold higher cancellation risk, especially at City Hotel. 

## 8. ADR Analysis

The Average Daily Rate (ADR) measures the average room rate that is associated with the booking. This analysis offers an examination of how ADR, hotel types, cancellation status, and arrival months all differ from one another. 

In [74]:
adr_by_hotel = (
    df.groupby("hotel")["adr"]
      .agg(["count", "mean", "median"])
      .sort_values("mean", ascending=False)
)

adr_by_hotel.round(2)

,count,mean,median
hotel,,,
City Hotel,79330,105.30,99.9
Resort Hotel,40060,94.95,75.0


In [75]:
adr_by_cancellation = (
    df.groupby(["hotel", "is_canceled"])["adr"]
      .agg(["count", "mean", "median"])
)

adr_by_cancellation.round(2)

count    mean  median
hotel        is_canceled                       
City Hotel   0            46228  105.75    99.9
             1            33102  104.69    99.8
Resort Hotel 0            28938   90.79    72.0
             1            11122  105.79    84.0

In [76]:
adr_by_month = (
    df.groupby("arrival_date_month")["adr"]
      .agg(["count", "mean", "median"])
      .sort_values("mean", ascending=False)
)

adr_by_month.round(2)

,count,mean,median
arrival_date_month,,,
August,13877,140.11,130.50
July,12661,126.79,118.10
June,10939,116.67,115.00
May,11791,108.70,108.00
September,10508,105.05,100.30
April,11089,100.38,96.30
October,11160,87.91,85.67
December,6780,81.08,75.00
March,9794,80.68,79.20


### ADR Analysis Finding 

The City Hotel has a higher overall average ADR than that of the Resort Hotel, which is $105 compared to $95.

The ADR is also varied by the month. We see that August has the highest average ADR at exactly $140, while in january had the lowest set for $70. This shows a difference of nearly $70 per room night and indicates strong seasonal variation in pricing. 

The cancellation status has little effect on City Hotel's ADR; However, Resort Hotel had noticeably higher ADR amongst then canceled bookings than completed bookings, suggesting a potential relationship between higher-priced reservations and cancellation behavior. 

## 9. Hotel Performance

Hotel-level performance was evaluated using the booking volume, Average Daily Rate (ADR) and cancellation rate. The comparison helps identify differences in demand, pricing, and booking reliability between City Hotel and Resort Hotel. 

In [77]:
hotel_performance = (
    df.groupby("hotel")
      .agg(
          bookings=("hotel", "count"),
          avg_adr=("adr", "mean"),
          median_adr=("adr", "median"),
          cancellation_rate=("is_canceled", "mean")
      )
      .sort_values("bookings", ascending=False)
)

hotel_performance.round(2)

,bookings,avg_adr,median_adr,cancellation_rate
hotel,,,,
City Hotel,79330,105.30,99.9,0.42
Resort Hotel,40060,94.95,75.0,0.28


In [78]:
monthly_bookings = (
    df.groupby(["arrival_date_month", "hotel"])
      .size()
      .unstack()
)

monthly_bookings

hotel,City Hotel,Resort Hotel
arrival_date_month,,
April,7480,3609
August,8983,4894
December,4132,2648
February,4965,3103
January,3736,2193
July,8088,4573
June,7894,3045
March,6458,3336
May,8232,3559


### Hotel Performance Finding 

The City Hotel had generated more bookings than the Resort Hotel, with approzimately 79,329 bookings compared to 40,059. City Hotel also appears to have a higher ADR.

However, City Hotel's cancellation rate was much higher, set at 41.7% compared to Resort Hotel, being 27.8%.

City Hotel also leads Resort Hotel in booking volume during every month in the dataset, with August representing the highest-volume month for both properties.

Overall, City Hotel shows to have a higher-volume, wi=hile there being low cancellations at Resort Hotel.

## 10. Booking Value Analysis

Estimated booking value was calculated from the Average Daily Rate (ADR) and the total number of nights booked:

ADR x (weekend nights + weekday nights)

This provides greater insight - an estimation of room value from each booking. This is not an official accounting revenue. 

In [79]:
df["estimated_booking_value"] = (
    df["adr"] *
    (
        df["stays_in_weekend_nights"] +
        df["stays_in_week_nights"]
    )
)

In [80]:
booking_value_by_hotel = (
    df.groupby("hotel")
      .agg(
          bookings=("hotel", "count"),
          estimated_booking_value=("estimated_booking_value", "sum"),
          avg_booking_value=("estimated_booking_value", "mean")
      )
      .sort_values("estimated_booking_value", ascending=False)
)

booking_value_by_hotel.round(2)

,bookings,estimated_booking_value,avg_booking_value
hotel,,,
City Hotel,79330,25279469.96,318.66
Resort Hotel,40060,17444027.57,435.45


In [81]:
completed_booking_value = (
    df[df["is_canceled"] == 0]
    .groupby("hotel")
    .agg(
        completed_bookings=("hotel", "count"),
        estimated_completed_value=("estimated_booking_value", "sum"),
        avg_completed_booking_value=("estimated_booking_value", "mean")
    )
    .sort_values("estimated_completed_value", ascending=False)
)

completed_booking_value.round(2)

,completed_bookings,estimated_completed_value,avg_completed_booking_value
hotel,,,
City Hotel,46228,14394410.18,311.38
Resort Hotel,28938,11601850.23,400.92


### Booking Value Finding

The City Hotel has generated a higher total estimated booking value set at $25.27 million, compared with $17.44 million for Resort Hotel. This difference was mainly die to City Hotel's higher booking volume.

However, Resort Hotel has a higher average estimated booking value, set at $435 per booking compared with the $319 for City Hotel.

When canceled bookings are excluded, City Hotel remained the highest in total estimated completed booking value at approximately $14.39 million, while Resort Hotel generated approximately $11.60 million. 

The results suggest that City Hotel's strength is booking volume, while Resort Hotel generates greater estimated value per booking. 

## 11. Monthly Performance

The monthly performance was analyzed to help identify any seasonal patterns in booking volume, cancellation behavior, and Average Daily Rate (ADR).

With these patterns, ther comes a better understanding to help identify periods of higher demand, pricing opportunities, and potential cancellation risk. 

In [82]:
monthly_booking_volume = (
    df.groupby("arrival_date_month")
      .size()
      .sort_values(ascending=False)
)

monthly_booking_volume

arrival_date_month
August       13877
July         12661
May          11791
October      11160
April        11089
June         10939
September    10508
March         9794
February      8068
November      6794
December      6780
January       5929
dtype: int64

In [83]:
monthly_cancellation_rate = (
    df.groupby("arrival_date_month")["is_canceled"]
      .mean()
      .sort_values(ascending=False)
)

monthly_cancellation_rate.round(3)

arrival_date_month
June         0.415
April        0.408
May          0.397
September    0.392
October      0.380
August       0.378
July         0.375
December     0.350
February     0.334
March        0.322
November     0.312
January      0.305
Name: is_canceled, dtype: float64

In [84]:
monthly_adr = (
    df.groupby("arrival_date_month")["adr"]
      .agg(["count", "mean", "median"])
      .sort_values("mean", ascending=False)
)

monthly_adr.round(2)

,count,mean,median
arrival_date_month,,,
August,13877,140.11,130.50
July,12661,126.79,118.10
June,10939,116.67,115.00
May,11791,108.70,108.00
September,10508,105.05,100.30
April,11089,100.38,96.30
October,11160,87.91,85.67
December,6780,81.08,75.00
March,9794,80.68,79.20


In [85]:
monthly_completed_value = (
    df[df["is_canceled"] == 0]
    .groupby("arrival_date_month")
    .agg(
        completed_bookings=("hotel", "count"),
        estimated_completed_value=("estimated_booking_value", "sum"),
        avg_booking_value=("estimated_booking_value", "mean")
    )
    .sort_values("estimated_completed_value", ascending=False)
)

monthly_completed_value.round(2)

,completed_bookings,estimated_completed_value,avg_booking_value
arrival_date_month,,,
August,8638,4917159.06,569.25
July,7919,4100397.07,517.79
June,6404,2583290.92,403.39
May,7114,2395213.64,336.69
September,6392,2344263.36,366.75
April,6565,2099857.48,319.86
October,6914,1856816.72,268.56
March,6645,1652098.59,248.62
February,5372,1147541.48,213.62


In [86]:
monthly_hotel_performance = (
    df[df["is_canceled"] == 0]
    .groupby(["arrival_date_month", "hotel"])
    .agg(
        completed_bookings=("hotel", "count"),
        estimated_completed_value=("estimated_booking_value", "sum"),
        avg_booking_value=("estimated_booking_value", "mean")
    )
    .sort_values("estimated_completed_value", ascending=False)
)

monthly_hotel_performance.round(2)

,,completed_bookings,estimated_completed_value,avg_booking_value
arrival_date_month,hotel,,,
August,Resort Hotel,3257,2939056.61,902.38
July,Resort Hotel,3137,2414119.49,769.56
August,City Hotel,5381,1978102.45,367.61
July,City Hotel,4782,1686277.58,352.63
May,City Hotel,4579,1569623.77,342.79
June,City Hotel,4366,1467868.67,336.20
April,City Hotel,4015,1369897.57,341.19
September,City Hotel,4290,1359321.31,316.86
October,City Hotel,4337,1241013.13,286.15


### Monthly Performance Finding

The booking activity showed strong seasonal variation. August has the highest booking volume set at 13,877 bookings, while January has the lowest at approximately 5,929. 

Pricing also varies throughout the year. In August, the highest ADR was $140, compared to $70 in January. 

Cancellation rates vary by month as well, like June and April having the highest cancallation rates. 

When the completed bookings were evaluted by estimated booking value, the strongest performance was set in the summer months, particularly July and August. Resort Hotel generated particularly high estimated completed-booking value during these months, whhile City Hotel maintained stronger booking volume across the year. 

Overall, the results indicate meaningful seasonality in both demand and pricing, with summer months representing an important period for hotel performance. 

## 12. Key Findings

The analysis has identified many important patterns in hotel booking performance:

- City Hotel had a much higher booking volume, with 79,329 bookings compared with 40,059 for Resort Hotel. 
- City Hotel had significantly higher cancellation rate, set at 41.7% compared with 27.8% for Resort Hotel. 
- Canceled bookings had longer lead times, averaging at 145 days compared with 80 for the complated bookings. 
- City Hotel had a higher overall ADR, approximately %105 compared with $95 for Resort Hotel.
- ADR showed a seasonality very strong, with August having a much higher average ADR at $140 and January the lowest at $70.
-City Hotel had generated more for total estimated booking value, exaclty $25.27 million, in comparison with $17.44 million for Resort Hotel. 
- Resort Hotel had generated a higher estimated value per booking, averaging at $435 compared to $319 with City Hotel. 
- Summer months showed the strongest overall performance, with July and August standing out for price, booking activity, and estimated completed-booking value.
- Agent activity was exceedingly highly concentrated, with top ten agents making up for 69.2% of the bookings with an agent assigned. 

All in all, the analysis suggests that the hotel performance has a combination of influence, deriving from booking value, cancellation behavior, pricing, booking value, and the season. City Hotel's strength was its higher volume, and Resort hotel demonstrated stronger value per booking and lower cancellation rates. 